# BirdCLEF 2026: Stage 2 Soundscape Adaptation + Stage 3 Pseudo-Labeling

This notebook implements two training stages in one self-contained notebook:

- **Stage 2**: fine-tune the species classifier on labeled 5-second soundscape windows.
- **Stage 3**: optionally pseudo-label high-confidence unlabeled soundscape windows, then fine-tune again.

There are no helper scripts. All model, dataset, training, validation, and pseudo-labeling code lives in this notebook.

Expected inputs from notebook 01:

- `birdclef_work/label_map.csv`
- `birdclef_work/train_soundscape_windows.csv`
- original dataset folder: `birdclef-2026/`

Optional input:

- `birdclef_work/checkpoints/stage1_train_audio_best.pt`

If the Stage 1 checkpoint does not exist, this notebook trains the same model architecture from scratch on soundscape labels.

## 0. Imports and Hardware Check

In [1]:
from pathlib import Path
import gc
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyTorch is required for this notebook. Activate the project .venv and install a CUDA-enabled PyTorch build first."
    ) from exc

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

try:
    from scipy.stats import rankdata
except Exception:
    rankdata = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE)
if torch.cuda.is_available():
    print("CUDA version used by PyTorch:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print("\n".join(result.stdout.splitlines()[:14]))
except Exception:
    pass

Python: 3.11.4
Platform: Windows-10-10.0.19045-SP0
Torch: 2.11.0+cu128
CUDA available: True
Selected device: cuda
CUDA version used by PyTorch: 12.8
GPU: NVIDIA GeForce RTX 3060
VRAM GB: 12.0
Tue Jun  2 14:09:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.49                 Driver Version: 596.49         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060      WDDM  |   00000000:01:00.0  On |                  N/A |
| 34%   45C    P8             25W /  170

## 1. Configuration

Keep Stage 3 disabled until Stage 2 finishes cleanly. Pseudo-labeling every unlabeled soundscape can take a long time, so the default Stage 3 settings process a limited number of files first.

In [2]:
PROJECT_DIR = Path.cwd()
WORK_DIR = PROJECT_DIR / "birdclef_work"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def find_dataset_dir():
    candidates = []
    env_dir = os.environ.get("BIRDCLEF_DATA_DIR")
    if env_dir:
        candidates.append(Path(env_dir))
    for base in [PROJECT_DIR, PROJECT_DIR.parent, Path("/kaggle/input")]:
        candidates.extend([base / "birdclef-2026", base / "birdcle-2026"])
    for candidate in candidates:
        if candidate.exists() and candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Could not find birdclef-2026 dataset folder.")

DATA_DIR = find_dataset_dir()

TARGET_SR = 32_000
CLIP_SECONDS = 5
CLIP_SAMPLES = TARGET_SR * CLIP_SECONDS
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
FMIN = 20
FMAX = TARGET_SR // 2

STAGE1_CHECKPOINT_PATH = CHECKPOINT_DIR / "stage1_train_audio_best.pt"
STAGE2_BEST_PATH = CHECKPOINT_DIR / "stage2_soundscape_best.pt"
STAGE2_LAST_PATH = CHECKPOINT_DIR / "stage2_soundscape_last.pt"
STAGE3_BEST_PATH = CHECKPOINT_DIR / "stage3_pseudolabel_best.pt"
STAGE3_LAST_PATH = CHECKPOINT_DIR / "stage3_pseudolabel_last.pt"

VAL_FRACTION = 0.20
BATCH_SIZE = 64 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0
STAGE2_EPOCHS = 12
STAGE2_LR = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 5.0

RUN_STAGE2 = True
RUN_STAGE3 = False
PSEUDO_MAX_FILES = 200
PSEUDO_THRESHOLD = 0.80
PSEUDO_TOP_K = 5
PSEUDO_MIN_ROWS = 50
STAGE3_EPOCHS = 4
STAGE3_LR = 8e-5

print("Dataset:", DATA_DIR)
print("Work dir:", WORK_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Batch size:", BATCH_SIZE)
print("Stage 1 checkpoint path:", STAGE1_CHECKPOINT_PATH)

Dataset: D:\BirdCLEF+\birdclef-2026
Work dir: d:\BirdCLEF+\birdclef_work
Checkpoint dir: d:\BirdCLEF+\birdclef_work\checkpoints
Batch size: 64
Stage 1 checkpoint path: d:\BirdCLEF+\birdclef_work\checkpoints\stage1_train_audio_best.pt


## 2. Load Prepared Manifests and Build Grouped Split

In [3]:
label_map_path = WORK_DIR / "label_map.csv"
soundscape_windows_path = WORK_DIR / "train_soundscape_windows.csv"

if not label_map_path.exists():
    raise FileNotFoundError(f"Missing {label_map_path}. Run notebook 01 first.")
if not soundscape_windows_path.exists():
    raise FileNotFoundError(f"Missing {soundscape_windows_path}. Run notebook 01 first.")

label_map = pd.read_csv(label_map_path, dtype={"primary_label": str})
target_labels = label_map["primary_label"].astype(str).tolist()
num_classes = len(target_labels)

soundscape_df = pd.read_csv(soundscape_windows_path, dtype={"filename": str})
soundscape_df["audio_path"] = soundscape_df["audio_path"].astype(str)
soundscape_df["start_seconds"] = soundscape_df["start_seconds"].astype(float)
soundscape_df[target_labels] = soundscape_df[target_labels].astype(np.float32)

missing_files = [p for p in soundscape_df["audio_path"].unique() if not Path(p).exists()]
if missing_files:
    raise FileNotFoundError(f"Some soundscape audio files are missing. First missing path: {missing_files[0]}")

unique_files = sorted(soundscape_df["filename"].unique())
rng = np.random.default_rng(SEED)
shuffled_files = unique_files.copy()
rng.shuffle(shuffled_files)
val_count = max(1, int(round(len(shuffled_files) * VAL_FRACTION)))
val_files = set(shuffled_files[:val_count])

train_df = soundscape_df[~soundscape_df["filename"].isin(val_files)].reset_index(drop=True)
val_df = soundscape_df[soundscape_df["filename"].isin(val_files)].reset_index(drop=True)

assert len(train_df) > 0 and len(val_df) > 0, "Train/validation split produced an empty split."
assert set(train_df["filename"]).isdisjoint(set(val_df["filename"])), "File leakage between train and validation splits."

split_path = WORK_DIR / "stage2_soundscape_split.csv"
split_export = soundscape_df[["filename", "start", "end", "start_seconds", "labels_string"]].copy()
split_export["split"] = np.where(split_export["filename"].isin(val_files), "val", "train")
split_export.to_csv(split_path, index=False)

print("Target classes:", num_classes)
print("Soundscape rows:", len(soundscape_df))
print("Train rows:", len(train_df), "files:", train_df["filename"].nunique())
print("Val rows:", len(val_df), "files:", val_df["filename"].nunique())
print("Saved split:", split_path)
print("Train positive classes:", int((train_df[target_labels].sum(axis=0) > 0).sum()))
print("Val positive classes:", int((val_df[target_labels].sum(axis=0) > 0).sum()))
train_df.head()

Target classes: 234
Soundscape rows: 1478
Train rows: 1180 files: 53
Val rows: 298 files: 13
Saved split: d:\BirdCLEF+\birdclef_work\stage2_soundscape_split.csv
Train positive classes: 72
Val positive classes: 33


,filename,audio_path,file_exists,start,end,start_seconds,end_seconds,duration_seconds,labels_string,positive_count,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,...,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:00,00:00:05,0.0,5,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:05,00:00:10,5.0,10,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:10,00:00:15,10.0,15,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:15,00:00:20,15.0,20,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:20,00:00:25,20.0,25,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Audio Loading and Log-Mel Functions

In [4]:
AUDIO_BACKEND = None

try:
    import librosa
    AUDIO_BACKEND = "librosa"
    print("Audio backend: librosa", librosa.__version__)
except Exception as librosa_exc:
    librosa = None
    try:
        import torchaudio
        AUDIO_BACKEND = "torchaudio"
        print("Audio backend: torchaudio", torchaudio.__version__)
    except Exception as torchaudio_exc:
        torchaudio = None
        raise ModuleNotFoundError(
            "This notebook needs either librosa+soundfile or torchaudio to read .ogg files. "
            "Install one backend in the selected notebook kernel before training."
        ) from torchaudio_exc

def pad_or_trim(y, target_len=CLIP_SAMPLES):
    y = np.asarray(y, dtype=np.float32)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    elif len(y) > target_len:
        y = y[:target_len]
    return y

def load_audio(path, offset_seconds=0.0, duration_seconds=CLIP_SECONDS, target_sr=TARGET_SR):
    path = Path(path)
    if AUDIO_BACKEND == "librosa":
        y, _ = librosa.load(path, sr=target_sr, mono=True, offset=float(offset_seconds), duration=float(duration_seconds))
        return pad_or_trim(y, int(target_sr * duration_seconds))

    waveform, sr = torchaudio.load(str(path))
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    start = int(round(float(offset_seconds) * sr))
    end = start + int(round(float(duration_seconds) * sr))
    waveform = waveform[:, start:end]
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
    return pad_or_trim(waveform.squeeze(0).numpy(), int(target_sr * duration_seconds))

def get_audio_duration(path):
    path = Path(path)
    if AUDIO_BACKEND == "librosa":
        return float(librosa.get_duration(path=str(path)))
    info = torchaudio.info(str(path))
    return float(info.num_frames / info.sample_rate)

if AUDIO_BACKEND == "torchaudio":
    MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
        sample_rate=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        f_min=FMIN,
        f_max=FMAX,
        power=2.0,
    )
    DB_TRANSFORM = torchaudio.transforms.AmplitudeToDB(stype="power")

def waveform_to_logmel(y):
    y = np.asarray(y, dtype=np.float32)
    if AUDIO_BACKEND == "librosa":
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=TARGET_SR,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            fmin=FMIN,
            fmax=FMAX,
            power=2.0,
        )
        logmel = librosa.power_to_db(mel, ref=np.max)
    else:
        waveform = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
        logmel = DB_TRANSFORM(MEL_TRANSFORM(waveform)).squeeze(0).numpy()
    mean = float(logmel.mean())
    std = float(logmel.std())
    return ((logmel - mean) / (std + 1e-6)).astype(np.float32)

def augment_waveform(y):
    gain = np.random.uniform(0.75, 1.25)
    y = y * gain
    if np.random.rand() < 0.35:
        noise = np.random.normal(0, np.random.uniform(0.001, 0.006), size=y.shape).astype(np.float32)
        y = y + noise
    return np.clip(y, -1.0, 1.0).astype(np.float32)

def seconds_to_hhmmss(seconds):
    seconds = int(round(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

Audio backend: librosa 0.11.0


## 4. Model, Dataset, Metrics, and Training Helpers

In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class BirdCLEFSmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            nn.MaxPool2d(2),
            ConvBlock(32, 64),
            nn.MaxPool2d(2),
            ConvBlock(64, 128),
            nn.MaxPool2d(2),
            ConvBlock(128, 256),
            nn.MaxPool2d(2),
            ConvBlock(256, 384),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.25)
        self.classifier = nn.Linear(384, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.classifier(x)

class SoundscapeWindowDataset(Dataset):
    def __init__(self, df, target_labels, augment=False):
        self.df = df.reset_index(drop=True)
        self.target_labels = target_labels
        self.paths = self.df["audio_path"].astype(str).values
        self.starts = self.df["start_seconds"].astype(float).values
        self.targets = self.df[target_labels].values.astype(np.float32)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        y = load_audio(self.paths[idx], offset_seconds=self.starts[idx], duration_seconds=CLIP_SECONDS)
        if self.augment:
            y = augment_waveform(y)
        x = waveform_to_logmel(y)
        x = torch.from_numpy(x).unsqueeze(0)
        target = torch.from_numpy(self.targets[idx])
        return x, target

def macro_auc_skip_empty(y_true, y_score):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_score = np.asarray(y_score, dtype=np.float32)
    aucs = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        ys = y_score[:, c]
        n_pos = int(yt.sum())
        n_neg = int(len(yt) - n_pos)
        if n_pos == 0 or n_neg == 0:
            continue
        if rankdata is not None:
            ranks = rankdata(ys)
        else:
            order = np.argsort(ys)
            ranks = np.empty_like(order, dtype=np.float32)
            ranks[order] = np.arange(1, len(ys) + 1)
        pos_rank_sum = ranks[yt == 1].sum()
        auc = (pos_rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
        aucs.append(float(auc))
    if not aucs:
        return float("nan"), 0
    return float(np.mean(aucs)), len(aucs)

def make_pos_weight(df):
    positives = df[target_labels].sum(axis=0).values.astype(np.float32)
    negatives = len(df) - positives
    pos_weight = np.ones_like(positives, dtype=np.float32)
    mask = positives > 0
    pos_weight[mask] = negatives[mask] / np.maximum(positives[mask], 1.0)
    pos_weight = np.clip(pos_weight, 1.0, 20.0)
    return torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)

def load_model_checkpoint(model, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        print(f"No checkpoint found at {checkpoint_path}. Training will start from initialized weights.")
        return False
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    cleaned = {}
    for key, value in state.items():
        cleaned[key.replace("module.", "")] = value
    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    print(f"Loaded checkpoint: {checkpoint_path}")
    print("Missing keys:", len(missing), "Unexpected keys:", len(unexpected))
    return True

def save_checkpoint(path, model, optimizer, epoch, best_auc, history, stage_name):
    payload = {
        "stage_name": stage_name,
        "epoch": int(epoch),
        "best_auc": float(best_auc) if best_auc == best_auc else None,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "target_labels": target_labels,
        "config": {
            "target_sr": TARGET_SR,
            "clip_seconds": CLIP_SECONDS,
            "n_mels": N_MELS,
            "hop_length": HOP_LENGTH,
            "n_fft": N_FFT,
        },
        "history": history,
    }
    torch.save(payload, path)

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    total_rows = 0
    for x, y in tqdm(loader, desc="train", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        if GRAD_CLIP_NORM is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        batch_size = x.size(0)
        total_loss += float(loss.detach().cpu()) * batch_size
        total_rows += batch_size
    return total_loss / max(total_rows, 1)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_rows = 0
    y_true_all = []
    y_score_all = []
    for x, y in tqdm(loader, desc="valid", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        probs = torch.sigmoid(logits)
        batch_size = x.size(0)
        total_loss += float(loss.detach().cpu()) * batch_size
        total_rows += batch_size
        y_true_all.append(y.detach().cpu().numpy())
        y_score_all.append(probs.detach().cpu().numpy())
    y_true = np.concatenate(y_true_all, axis=0)
    y_score = np.concatenate(y_score_all, axis=0)
    auc, used_classes = macro_auc_skip_empty(y_true, y_score)
    return total_loss / max(total_rows, 1), auc, used_classes

def make_loaders(stage_train_df, stage_val_df, batch_size=BATCH_SIZE):
    train_ds = SoundscapeWindowDataset(stage_train_df, target_labels, augment=True)
    val_ds = SoundscapeWindowDataset(stage_val_df, target_labels, augment=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    return train_loader, val_loader

def run_training_stage(model, stage_train_df, stage_val_df, epochs, lr, best_path, last_path, stage_name):
    train_loader, val_loader = make_loaders(stage_train_df, stage_val_df)
    pos_weight = make_pos_weight(stage_train_df)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    history = []
    best_auc = -1.0

    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss, val_auc, used_classes = evaluate(model, val_loader, criterion)
        elapsed = time.time() - start
        row = {
            "stage": stage_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_macro_auc": val_auc,
            "auc_classes_used": used_classes,
            "seconds": elapsed,
        }
        history.append(row)
        print(
            f"{stage_name} epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
            f"val_auc={val_auc:.5f} classes={used_classes} time={elapsed/60:.1f}m"
        )
        if val_auc == val_auc and val_auc > best_auc:
            best_auc = val_auc
            save_checkpoint(best_path, model, optimizer, epoch, best_auc, history, stage_name)
            print("Saved best checkpoint:", best_path)
        save_checkpoint(last_path, model, optimizer, epoch, best_auc, history, stage_name)
        pd.DataFrame(history).to_csv(WORK_DIR / f"{stage_name}_history.csv", index=False)
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return history, best_auc

## 5. Smoke Test Before Long Training

This catches audio loading, spectrogram shape, model shape, and loss-shape problems before a long training run starts.

In [6]:
model = BirdCLEFSmallCNN(num_classes=num_classes).to(DEVICE)
loaded_stage1 = load_model_checkpoint(model, STAGE1_CHECKPOINT_PATH)

smoke_df = train_df.sample(min(2, len(train_df)), random_state=SEED).reset_index(drop=True)
smoke_ds = SoundscapeWindowDataset(smoke_df, target_labels, augment=False)
smoke_x, smoke_y = smoke_ds[0]
print("One sample x/y:", tuple(smoke_x.shape), tuple(smoke_y.shape))

smoke_loader = DataLoader(smoke_ds, batch_size=len(smoke_ds), shuffle=False, num_workers=0)
x_batch, y_batch = next(iter(smoke_loader))
x_batch = x_batch.to(DEVICE)
y_batch = y_batch.to(DEVICE)
with torch.no_grad():
    smoke_logits = model(x_batch)
    smoke_loss = nn.BCEWithLogitsLoss()(smoke_logits, y_batch)
print("Batch x/y/logits:", tuple(x_batch.shape), tuple(y_batch.shape), tuple(smoke_logits.shape))
print("Smoke loss:", float(smoke_loss.detach().cpu()))
assert smoke_logits.shape == y_batch.shape, "Model output shape must match target shape."
print("Smoke test passed. Stage 2 training can start.")

No checkpoint found at d:\BirdCLEF+\birdclef_work\checkpoints\stage1_train_audio_best.pt. Training will start from initialized weights.
One sample x/y: (1, 128, 313) (234,)
Batch x/y/logits: (2, 1, 128, 313) (2, 234) (2, 234)
Smoke loss: 0.7027320861816406
Smoke test passed. Stage 2 training can start.


## 6. Stage 2: Soundscape Adaptation Training

In [7]:
if RUN_STAGE2:
    stage2_history, stage2_best_auc = run_training_stage(
        model=model,
        stage_train_df=train_df,
        stage_val_df=val_df,
        epochs=STAGE2_EPOCHS,
        lr=STAGE2_LR,
        best_path=STAGE2_BEST_PATH,
        last_path=STAGE2_LAST_PATH,
        stage_name="stage2_soundscape",
    )
    print("Stage 2 best validation AUC:", stage2_best_auc)
else:
    print("RUN_STAGE2 is False. Skipping Stage 2 training.")

stage2_soundscape epoch 01/12 | train_loss=0.7017 val_loss=0.7699 val_auc=0.77173 classes=33 time=0.5m
Saved best checkpoint: d:\BirdCLEF+\birdclef_work\checkpoints\stage2_soundscape_best.pt
stage2_soundscape epoch 02/12 | train_loss=0.5542 val_loss=0.6369 val_auc=0.82211 classes=33 time=0.4m
Saved best checkpoint: d:\BirdCLEF+\birdclef_work\checkpoints\stage2_soundscape_best.pt
stage2_soundscape epoch 03/12 | train_loss=0.4450 val_loss=0.4759 val_auc=0.86006 classes=33 time=0.4m
Saved best checkpoint: d:\BirdCLEF+\birdclef_work\checkpoints\stage2_soundscape_best.pt
stage2_soundscape epoch 04/12 | train_loss=0.3622 val_loss=0.3910 val_auc=0.87014 classes=33 time=0.4m
Saved best checkpoint: d:\BirdCLEF+\birdclef_work\checkpoints\stage2_soundscape_best.pt
stage2_soundscape epoch 05/12 | train_loss=0.2948 val_loss=0.3199 val_auc=0.87929 classes=33 time=0.4m
Saved best checkpoint: d:\BirdCLEF+\birdclef_work\checkpoints\stage2_soundscape_best.pt
stage2_soundscape epoch 06/12 | train_loss=0.

## 7. Stage 3 Pseudo-Labeling Functions

Stage 3 uses the Stage 2 model to score unlabeled `train_soundscapes/` windows. High-confidence predictions become soft targets for another fine-tuning pass.

In [8]:
def build_unlabeled_soundscape_manifest(max_files=PSEUDO_MAX_FILES):
    soundscape_dir = DATA_DIR / "train_soundscapes"
    all_files = sorted(soundscape_dir.glob("*.ogg"))
    labeled_files = set(soundscape_df["filename"].unique())
    unlabeled_files = [p for p in all_files if p.name not in labeled_files]
    rng = np.random.default_rng(SEED)
    unlabeled_files = list(unlabeled_files)
    rng.shuffle(unlabeled_files)
    if max_files is not None:
        unlabeled_files = unlabeled_files[:max_files]

    rows = []
    for path in tqdm(unlabeled_files, desc="scan unlabeled files"):
        try:
            duration = get_audio_duration(path)
        except Exception as exc:
            print("Skipping unreadable file:", path, repr(exc))
            continue
        n_windows = int(duration // CLIP_SECONDS)
        for w in range(n_windows):
            start = w * CLIP_SECONDS
            end = start + CLIP_SECONDS
            rows.append({
                "filename": path.name,
                "audio_path": str(path),
                "start_seconds": float(start),
                "end_seconds": float(end),
                "start": seconds_to_hhmmss(start),
                "end": seconds_to_hhmmss(end),
            })
    manifest = pd.DataFrame(rows)
    print("Unlabeled candidate windows:", len(manifest), "from files:", manifest["filename"].nunique() if len(manifest) else 0)
    return manifest

class InferenceWindowDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.paths = self.df["audio_path"].astype(str).values
        self.starts = self.df["start_seconds"].astype(float).values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        y = load_audio(self.paths[idx], offset_seconds=self.starts[idx], duration_seconds=CLIP_SECONDS)
        x = waveform_to_logmel(y)
        return torch.from_numpy(x).unsqueeze(0)

@torch.no_grad()
def generate_pseudo_labels(model, candidate_df, threshold=PSEUDO_THRESHOLD, top_k=PSEUDO_TOP_K):
    if len(candidate_df) == 0:
        return pd.DataFrame()
    ds = InferenceWindowDataset(candidate_df)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    model.eval()
    kept_rows = []
    offset = 0
    for x in tqdm(loader, desc="pseudo-label inference"):
        x = x.to(DEVICE, non_blocking=True)
        probs = torch.sigmoid(model(x)).detach().cpu().numpy()
        batch_meta = candidate_df.iloc[offset:offset + len(probs)].reset_index(drop=True)
        offset += len(probs)
        for i in range(len(probs)):
            scores = probs[i]
            positive_idx = np.where(scores >= threshold)[0]
            if len(positive_idx) == 0:
                continue
            if top_k is not None and len(positive_idx) > top_k:
                positive_idx = positive_idx[np.argsort(scores[positive_idx])[-top_k:]]
            target = np.zeros(num_classes, dtype=np.float32)
            target[positive_idx] = scores[positive_idx]
            labels_string = ";".join([target_labels[j] for j in positive_idx[np.argsort(scores[positive_idx])[::-1]]])
            meta = batch_meta.iloc[i].to_dict()
            meta.update({
                "source": "pseudo",
                "labels_string": labels_string,
                "pseudo_label_count": int(len(positive_idx)),
                "max_confidence": float(scores[positive_idx].max()),
            })
            for j, label in enumerate(target_labels):
                meta[label] = float(target[j])
            kept_rows.append(meta)
    pseudo_df = pd.DataFrame(kept_rows)
    print("Pseudo-labeled rows kept:", len(pseudo_df))
    return pseudo_df

def prepare_hard_labeled_for_stage3(df):
    cols = ["filename", "audio_path", "start", "end", "start_seconds", "end_seconds", "labels_string"] + target_labels
    hard = df.copy()
    if "end_seconds" not in hard.columns:
        hard["end_seconds"] = hard["start_seconds"] + CLIP_SECONDS
    hard = hard[cols].copy()
    hard["source"] = "labeled"
    hard["pseudo_label_count"] = hard[target_labels].sum(axis=1).astype(int)
    hard["max_confidence"] = 1.0
    return hard

print("Stage 3 functions are ready. RUN_STAGE3 is", RUN_STAGE3)

Stage 3 functions are ready. RUN_STAGE3 is False


## 8. Stage 3: Optional Pseudo-Label Fine-Tuning

To run this stage, set `RUN_STAGE3 = True` in the configuration cell and rerun from there. Start with `PSEUDO_MAX_FILES = 200`; after it works, increase the number or set it to `None` for all unlabeled soundscapes.

In [9]:
if RUN_STAGE3:
    if STAGE2_BEST_PATH.exists():
        load_model_checkpoint(model, STAGE2_BEST_PATH)
        model.to(DEVICE)
    else:
        print("Stage 2 best checkpoint is missing. Using the current in-memory model for pseudo-labeling.")

    candidate_df = build_unlabeled_soundscape_manifest(max_files=PSEUDO_MAX_FILES)
    candidate_path = WORK_DIR / "stage3_unlabeled_candidates.csv"
    candidate_df.to_csv(candidate_path, index=False)
    print("Saved candidates:", candidate_path)

    pseudo_df = generate_pseudo_labels(model, candidate_df, threshold=PSEUDO_THRESHOLD, top_k=PSEUDO_TOP_K)
    pseudo_path = WORK_DIR / "stage3_pseudo_labels.csv"
    pseudo_df.to_csv(pseudo_path, index=False)
    print("Saved pseudo labels:", pseudo_path)

    if len(pseudo_df) < PSEUDO_MIN_ROWS:
        print(f"Only {len(pseudo_df)} pseudo rows were kept, below PSEUDO_MIN_ROWS={PSEUDO_MIN_ROWS}. Skipping Stage 3 training.")
    else:
        hard_train = prepare_hard_labeled_for_stage3(train_df)
        stage3_cols = ["filename", "audio_path", "start", "end", "start_seconds", "end_seconds", "labels_string", "source", "pseudo_label_count", "max_confidence"] + target_labels
        pseudo_train = pseudo_df[stage3_cols].copy()
        combined_train = pd.concat([hard_train[stage3_cols], pseudo_train], ignore_index=True)
        combined_train[target_labels] = combined_train[target_labels].astype(np.float32)
        combined_path = WORK_DIR / "stage3_combined_train_windows.csv"
        combined_train.to_csv(combined_path, index=False)
        print("Combined Stage 3 rows:", len(combined_train))
        print("Saved combined training manifest:", combined_path)

        stage3_history, stage3_best_auc = run_training_stage(
            model=model,
            stage_train_df=combined_train,
            stage_val_df=val_df,
            epochs=STAGE3_EPOCHS,
            lr=STAGE3_LR,
            best_path=STAGE3_BEST_PATH,
            last_path=STAGE3_LAST_PATH,
            stage_name="stage3_pseudolabel",
        )
        print("Stage 3 best validation AUC:", stage3_best_auc)
else:
    print("RUN_STAGE3 is False. Stage 3 implementation is present but not running.")

RUN_STAGE3 is False. Stage 3 implementation is present but not running.


## Outputs

Stage 2 outputs:

- `birdclef_work/checkpoints/stage2_soundscape_best.pt`
- `birdclef_work/checkpoints/stage2_soundscape_last.pt`
- `birdclef_work/stage2_soundscape_history.csv`
- `birdclef_work/stage2_soundscape_split.csv`

Stage 3 outputs, if enabled:

- `birdclef_work/stage3_unlabeled_candidates.csv`
- `birdclef_work/stage3_pseudo_labels.csv`
- `birdclef_work/stage3_combined_train_windows.csv`
- `birdclef_work/checkpoints/stage3_pseudolabel_best.pt`
- `birdclef_work/checkpoints/stage3_pseudolabel_last.pt`
- `birdclef_work/stage3_pseudolabel_history.csv`

The next notebook should use the best checkpoint for hidden test soundscape inference and write `submission.csv`.